In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/kgvsfcekman/ekman6_vsfc_2.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_4.rdf
/kaggle/input/kgvsfcekman/ekman6_2_vsfc_3.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_weighted.rdf
/kaggle/input/kgvsfcekman/ekman6_4_4.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc_3.rdf
/kaggle/input/kgvsfcekman/ekman6_4_6.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_5.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_4.rdf
/kaggle/input/kgvsfcekman/ekman6_3_vsfc_1.rdf
/kaggle/input/kgvsfcekman/ekman6_4_3.rdf
/kaggle/input/kgvsfcekman/ekman6_vsfc.rdf
/kaggle/input/vsfcekman-1/dev.csv
/kaggle/input/vsfcekman-1/train.csv
/kaggle/input/vsfcekman-1/test.csv
/kaggle/input/kg-ekman6/ekman6_4_5.rdf
/kaggle/input/kg-ekman6/ekman6_4_7.rdf
/kaggle/input/kg-ekman6/ekman6_4_4.rdf
/kaggle/input/kg-ekman6/ekman6_4_6.rdf
/kaggle/input/kg-ekman6/ekman6_4_8.rdf


In [2]:
# ---------------------------------------------------------
# CELL 1: SETUP & DEPENDENCIES
# ---------------------------------------------------------
!pip install -q pyvi rdflib unidecode

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import copy
import re
import rdflib
from rdflib import Graph, Namespace, RDF, RDFS
from pyvi import ViTokenizer
from unidecode import unidecode
import os
import random

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 59.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.6 MB/s eta 0:00:0000:01
✅ Using device: cuda


In [3]:
# =========================================================
# CELL 2: ONTOLOGY ENGINE V12.2 (VSFC EDITION)
# Logic: Deep Inference (Suy diễn sâu) + Soft Polarity.
# Target: Tối ưu cho bài toán Sentiment 3 nhãn (Pos/Neg/Neu).
# Input: ekman6_4_8.rdf
# =========================================================
!pip install -q pyvi rdflib unidecode

import os, re
import numpy as np
import rdflib
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL
from pyvi import ViTokenizer
from unidecode import unidecode

class OntologyEngine:
    def __init__(self, rdfpath, defaultconf=0.85, manual_boost=1.2, negation_attenuation=0.5, alpha_similarity=0.2, verbose=True):
        self.EKMAN = Namespace("http://example.org/ekman-ontology#")
        self.defaultconf = float(defaultconf)
        self.manual_boost = float(manual_boost)
        self.negation_attenuation = float(negation_attenuation) 
        self.alpha_similarity = float(alpha_similarity)
        self.verbose = bool(verbose)

        # 1. CONFIGS
        self.ALLEMOTIONS = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
        self.ALLAPPRAISALS = [
            "GoalObstructionAppraisal", "PleasantnessAppraisal", "UnpleasantnessAppraisal", 
            "DangerousnessAppraisal", "LossAppraisal", "SuddennessAppraisal", 
            "LowPredictabilityAppraisal", "UnpredictabilityAppraisal", "UnfairnessAppraisal"
        ]
        self.INTENSITY_CLASSES = ["HighIntensity", "MediumIntensity", "LowIntensity"]
        
        # VSFC Rất cần 3 cột này chuẩn xác
        self.POLARITY_CLASSES = ["PositivePolarity", "NegativePolarity", "NeutralPolarity"]

        self.EMOTIONMAPPING = {
            "Enjoyment": "Happiness", "Happiness": "Happiness", "Anger": "Anger", "Disgust": "Disgust", 
            "Fear": "Fear", "Sadness": "Sadness", "Surprise": "Surprise", 
            "NeutralEmotion": "Neutral", "Neutral": "Neutral", "Other": "Neutral", "NegationCue": "NegationCue"
        }
        
        # [QUAN TRỌNG VỚI VSFC] Suy diễn Polarity nếu thiếu
        # Vì VSFC là bài toán Sentiment, việc map Emotion -> Polarity là bắt buộc
        self.IMPLIED_POLARITY = {
            # Emotion -> Polarity
            "Anger": "NegativePolarity", "Disgust": "NegativePolarity",
            "Fear": "NegativePolarity", "Sadness": "NegativePolarity",
            "Happiness": "PositivePolarity", "Enjoyment": "PositivePolarity",
            "Surprise": "PositivePolarity", 
            # Appraisal -> Polarity
            "UnpleasantnessAppraisal": "NegativePolarity",
            "GoalObstructionAppraisal": "NegativePolarity", 
            "LossAppraisal": "NegativePolarity",
            "UnfairnessAppraisal": "NegativePolarity",
            "DangerousnessAppraisal": "NegativePolarity",
            "PleasantnessAppraisal": "PositivePolarity"
        }
        
        self.AMBIGUOUS_TERMS = {
            "hay", "chả", "cơ", "ngờ", "ý", "quá", "lại", 
            "tôi", "tao", "tớ", "mình", "bạn", "nó", "hắn", "anh", "chị", "em",
            "là", "thì", "mà", "bị", "được", "cái", "con", "người"
        }
        
        self.VALID_DOMAINS = {"GeneralDomain", "SocialDomain", "EducationDomain"}
        
        # LOAD KG
        if self.verbose: print(f"📥 Loading KG for VSFC: {rdfpath}")
        self.g = Graph()
        if os.path.exists(rdfpath):
            self.g.parse(rdfpath)
            if self.verbose: print(f"   ✅ Loaded {len(self.g)} triples.")
        else:
            print(f"   ⚠️ ERROR: File not found {rdfpath}")

        self.sim_matrix = self._build_similarity_matrix()
        self.lexiconmap = self._build_strict_lexicon()
        
        self.mwe_max_len = 1
        for k in self.lexiconmap.keys(): self.mwe_max_len = max(self.mwe_max_len, len(k.split()))
        if self.verbose: print(f"   📚 Lexicon size: {len(self.lexiconmap)} entries (v12.2 VSFC-Ready)")

    def _norm(self, s): return re.sub(r"\s+", " ", str(s).lower().strip().replace("_", " ")).strip() if s else ""
    def _local(self, uri): return str(uri).split("#")[-1] if "#" in str(uri) else str(uri).split("/")[-1]

    def _build_similarity_matrix(self):
        n = len(self.ALLEMOTIONS)
        mat = np.zeros((n, n), dtype=np.float32); np.fill_diagonal(mat, 1.0)
        try:
            q = """PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?score ?e1 ?e2 WHERE { ?edge a ekman:EmotionSimilarityEdge ; ekman:similarityScore ?score ; ekman:linksEmotion ?e1 ; ekman:linksEmotion ?e2 . }"""
            for row in self.g.query(q):
                e1, e2 = self.EMOTIONMAPPING.get(self._local(row.e1)), self.EMOTIONMAPPING.get(self._local(row.e2))
                if e1 in self.ALLEMOTIONS and e2 in self.ALLEMOTIONS:
                    i, j = self.ALLEMOTIONS.index(e1), self.ALLEMOTIONS.index(e2)
                    mat[i, j] = mat[j, i] = max(mat[i, j], float(row.score))
        except: pass
        s = mat.sum(axis=1, keepdims=True); s[s==0] = 1.0
        return mat / s

    def _build_strict_lexicon(self):
        EK, lookup = self.EKMAN, {}
        mix_cache, sim_cache = {}, {}
        
        for r in self.g.query("PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?m ?e ?w WHERE { ?m a ekman:EmotionMixture ; ekman:hasComponent ?c . ?c ekman:componentEmotion ?e ; ekman:componentWeight ?w . }"):
            m, e, w = r.m, self.EMOTIONMAPPING.get(self._local(r.e)), float(r.w)
            if e in self.ALLEMOTIONS: mix_cache.setdefault(m, {})[e] = mix_cache.setdefault(m, {}).get(e, 0.0) + w
        
        for s, _, o in self.g.triples((None, EK.hasEmotionCategory, None)):
            e = self.EMOTIONMAPPING.get(self._local(o))
            if e in self.ALLEMOTIONS: sim_cache[s] = [e]
            
        for s in self.g.subjects(RDF.type, None):
            if s in mix_cache or s in sim_cache: continue
            ts = list(self.g.objects(s, RDF.type))
            if EK.NegationCue in ts: sim_cache[s] = ["NegationCue"]
            else:
                for t in ts:
                    m = self.EMOTIONMAPPING.get(self._local(t))
                    if m in self.ALLEMOTIONS: sim_cache[s] = [m]; break
        
        q = """PREFIX ekman: <http://example.org/ekman-ontology#> SELECT ?word ?stim ?ev ?base ?type ?dom WHERE { ?evoc a ekman:Evocation ; ekman:fromLexiconEntry ?lex ; ekman:toStimulus ?stim . ?lex ekman:hasContent ?word . OPTIONAL { ?evoc ekman:confidenceScore ?ev } OPTIONAL { ?evoc ekman:evidenceType ?type } OPTIONAL { ?lex ekman:baseIntensityScore ?base } OPTIONAL { ?lex ekman:belongsToDomain ?dom } }"""
        for r in self.g.query(q):
            if r.dom and self._local(r.dom) not in self.VALID_DOMAINS: continue 
            w = self._norm(r.word)
            if len(w) < 2 or w in self.AMBIGUOUS_TERMS: continue
            
            emos = {}
            if r.stim in mix_cache:
                raw = mix_cache[r.stim]; tot = sum(raw.values())
                if tot > 0: emos = {k: v/tot for k, v in raw.items()}
            elif r.stim in sim_cache:
                lst = sim_cache[r.stim]
                emos = {k: 1.0 for k in lst}
            if not emos: continue

            # Deep Inference
            apps, ints, pols = self._infer_attributes(r.stim, emos.keys())
            
            entry_score = float(r.ev or self.defaultconf) * float(r.base or 1.0)
            
            entry = {"emotions": emos, "score": entry_score, 
                     "appraisals": list(apps), "intensities": list(ints), "polarities": list(pols), 
                     "isnegation": "NegationCue" in emos}
            lookup.setdefault(w, []).append(entry)
            wu = self._norm(unidecode(w))
            if wu != w: lookup.setdefault(wu, []).append(entry)
        return lookup

    def _infer_attributes(self, s, emos):
        EK, a, i, p = self.EKMAN, set(), set(), set()
        
        # 1. Direct
        def g(r):
            for o in self.g.objects(r, EK.evokesAppraisal): a.add(self._local(o))
            for o in self.g.objects(r, EK.hasIntensity): i.add(self._local(o))
            for o in self.g.objects(r, EK.hasPolarity): p.add(self._local(o))
        g(s)
        for t in self.g.objects(s, RDF.type): g(t)
        
        # 2. Deep Inference (Quan trọng cho VSFC)
        if not p:
            for e in emos:
                if e in self.IMPLIED_POLARITY: p.add(self.IMPLIED_POLARITY[e])
        if not p:
            for app in a:
                if app in self.IMPLIED_POLARITY: p.add(self.IMPLIED_POLARITY[app])

        return a, i, p

    def getvector(self, text, debug=False):
        tokens = ViTokenizer.tokenize(str(text).lower()).split()
        flat = [self._norm(t) for t in tokens if self._norm(t)]
        
        ve, va, vi = np.zeros(7, dtype=np.float32), np.zeros(9, dtype=np.float32), np.zeros(3, dtype=np.float32)
        vp, vn = np.zeros(3, dtype=np.float32), np.zeros(2, dtype=np.float32)
        
        idx, n = 0, len(flat)
        hit_trace = []

        while idx < n:
            m, ml = None, 0
            for L in range(min(self.mwe_max_len, n-idx), 0, -1):
                p = " ".join(flat[idx:idx+L])
                if p in self.lexiconmap: m, ml = self.lexiconmap[p], L; break
            
            if not m:
                idx += 1; continue
            
            if debug: hit_trace.append({"phrase": " ".join(flat[idx:idx+ml]), "data": m[0]})

            for ent in m:
                if ent["isnegation"]:
                    vn[0] = 1.0; continue
                
                sc = ent["score"]
                if vn[0] > 0: sc *= self.negation_attenuation 
                
                for e, w in ent["emotions"].items():
                    if e in self.ALLEMOTIONS: ve[self.ALLEMOTIONS.index(e)] += sc * w
                for x in ent["appraisals"]:
                    if x in self.ALLAPPRAISALS: va[self.ALLAPPRAISALS.index(x)] += sc
                for x in ent["intensities"]:
                    if x in self.INTENSITY_CLASSES: vi[self.INTENSITY_CLASSES.index(x)] = 1.0
                
                # Soft Polarity (Quan trọng)
                for x in ent["polarities"]:
                    if x in self.POLARITY_CLASSES: vp[self.POLARITY_CLASSES.index(x)] += sc 
            idx += ml

        if self.alpha_similarity > 0: ve = np.clip(ve + self.alpha_similarity * (ve @ self.sim_matrix), 0, None)
        fin = np.concatenate([np.tanh(ve), np.tanh(va), vi, vp, vn]).astype(np.float32)
        return (fin, flat, hit_trace) if debug else fin

# KHỞI TẠO (Dùng file 6_4_8)
RDFPATH = "/kaggle/input/kg-ekman6/ekman6_4_8.rdf"
if os.path.exists(RDFPATH):
    print(f"🚀 Initializing OntologyEngine v12.2 for VSFC (Deep Inference) with {RDFPATH}...")
    ontengine = OntologyEngine(RDFPATH, verbose=True)
else:
    print(f"❌ ERROR: File not found {RDFPATH}")

🚀 Initializing OntologyEngine v12.2 for VSFC (Deep Inference) with /kaggle/input/kg-ekman6/ekman6_4_8.rdf...
📥 Loading KG for VSFC: /kaggle/input/kg-ekman6/ekman6_4_8.rdf
   ✅ Loaded 90420 triples.
   📚 Lexicon size: 2540 entries (v12.2 VSFC-Ready)


In [4]:
# ---------------------------------------------------------
# CELL 3: DATASET & DATALOADER
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, ontology_engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.ont_engine = ontology_engine
        self.max_len = max_len
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        raw_text = str(self.df.iloc[index]['Sentence'])
        label = self.df.iloc[index]['label_idx']
        
        encoding = self.tokenizer.encode_plus(
            raw_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        # Gọi hàm getvector (đã sửa tên đúng)
        ont_vector = self.ont_engine.getvector(raw_text) 
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'ontology_features': torch.tensor(ont_vector, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Load Data
try:
    base_path = "/kaggle/input/vsfcekman-1" # Check path
    df_train = pd.read_csv(f"{base_path}/train.csv")
    df_valid = pd.read_csv(f"{base_path}/dev.csv")
    df_test = pd.read_csv(f"{base_path}/test.csv")
    
    for df in [df_train, df_valid, df_test]:
        df.rename(columns={'Text': 'Sentence', 'Ekman_Label': 'Emotion'}, inplace=True)
        df.dropna(subset=['Sentence', 'Emotion'], inplace=True)
    
    print("✅ Loaded Datasets.")
except Exception as e:
    print(f"❌ Error loading data: {e}")

# Label Encoding
le = LabelEncoder()
all_labels = sorted(df_train['Emotion'].unique().astype(str))
le.fit(all_labels)
print(f"📋 Classes: {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df['label_idx'] = le.transform(df['Emotion'].astype(str))

# Dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(EmotionDataset(df_train, tokenizer, ontengine), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(EmotionDataset(df_valid, tokenizer, ontengine), batch_size=BATCH_SIZE)
test_loader = DataLoader(EmotionDataset(df_test, tokenizer, ontengine), batch_size=BATCH_SIZE)

# Class Weights
train_labels = df_train['label_idx'].tolist()
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"⚖️ Weights: {weights_tensor}")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Loaded Datasets.
📋 Classes: ['Anger' 'Disgust' 'Fear' 'Happiness' 'Neutral' 'Sadness' 'Surprise']
⚖️ Weights: tensor([ 0.7617,  2.6327, 41.8535,  0.2899,  3.5717,  0.6545, 38.8639],
       device='cuda:0')


In [5]:
# ---------------------------------------------------------
# CELL 4: MODEL (xAI & DENSE SUPPORT)
# ---------------------------------------------------------
try: ONT_INPUT_DIM = len(ontengine.getvector("t"))
except: ONT_INPUT_DIM = 24

class PhoBERT_Fusion_V2(nn.Module):
    def __init__(self, n_classes, fusion_type='none', ontology_dim=None, ont_input_dim=ONT_INPUT_DIM):
        super(PhoBERT_Fusion_V2, self).__init__()
        self.fusion_type = fusion_type
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # Ontology Adapter
        if ontology_dim: # Dense Mode (64d)
            self.ont_adapter = nn.Sequential(
                nn.Linear(ont_input_dim, ontology_dim),
                nn.BatchNorm1d(ontology_dim),
                nn.ReLU(),
                nn.Dropout(0.1)
            )
            self.curr_dim = ontology_dim
        else: # Raw Mode (xAI)
            self.ont_adapter = nn.Identity()
            self.curr_dim = ont_input_dim
            
        # Fusion
        if self.fusion_type == 'gate':
            self.gate = nn.Linear(768, self.curr_dim)
            
        # Head
        inp = 768 + self.curr_dim if self.fusion_type != 'none' else 768
        self.fc = nn.Linear(inp, n_classes)

    def forward(self, ids, mask, ont):
        _, txt = self.phobert(ids, mask, return_dict=False)
        txt = self.dropout(txt)
        
        if self.fusion_type == 'none': return self.fc(txt)
        
        ont_emb = self.ont_adapter(ont)
        if self.fusion_type == 'gate':
            g = torch.sigmoid(self.gate(txt))
            ont_emb = ont_emb * g
            
        comb = torch.cat([txt, ont_emb], dim=1)
        return self.fc(comb)

In [6]:
# ---------------------------------------------------------
# CELL 5: TRAINING HELPER FUNCTIONS (EARLY STOPPING)
# ---------------------------------------------------------
from sklearn.metrics import accuracy_score, f1_score

def train_experiment(model, name, epochs=20, patience=3):
    print(f"\n🚀 Bắt đầu huấn luyện: {name}")
    print(f"   (Cấu hình: Max Epochs={epochs}, Patience={patience})")

    # Bọc DataParallel nếu GPU > 1
    if torch.cuda.device_count() > 1:
        print(f"⚡ Đang chạy trên {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)
    
    best_f1 = 0
    # Copy weights ban đầu
    if isinstance(model, nn.DataParallel):
        best_model_wts = copy.deepcopy(model.module.state_dict())
    else:
        best_model_wts = copy.deepcopy(model.state_dict())
        
    patience_counter = 0 
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            ont = batch['ontology_features'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, mask, ont)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Đánh giá sau mỗi epoch
        val_acc, val_f1 = evaluate(model, valid_loader)
        avg_loss = total_loss / len(train_loader)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}", end=" ")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            if isinstance(model, nn.DataParallel):
                best_model_wts = copy.deepcopy(model.module.state_dict())
            else:
                best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0 
            print("✅ New Best!")
        else:
            patience_counter += 1
            print(f"⚠️ No improve ({patience_counter}/{patience})")
            
        if patience_counter >= patience:
            print(f"\n🛑 Early Stopping triggered! (Không cải thiện sau {patience} epochs)")
            break
            
    print(f"🏆 Kết thúc {name}. Best Val F1: {best_f1:.4f}")
    
    # Load lại best weights trước khi trả về
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_model_wts)
    else:
        model.load_state_dict(best_model_wts)
        
    return model

def evaluate(model, loader):
    model.eval()
    preds = []
    truths = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            ont = batch['ontology_features'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, mask, ont)
            _, pred = torch.max(outputs, dim=1)
            
            preds.extend(pred.cpu().numpy())
            truths.extend(labels.cpu().numpy())
            
    return accuracy_score(truths, preds), f1_score(truths, preds, average='macro')

In [7]:
import pandas as pd

basepath = "/kaggle/input/vsfcekman-1"

dftrain = pd.read_csv(f"{basepath}/train.csv")
dfvalid = pd.read_csv(f"{basepath}/dev.csv")
dftest  = pd.read_csv(f"{basepath}/test.csv")

# chuẩn hoá tên cột theo notebook của bạn
for df in [dftrain, dfvalid, dftest]:
    df.rename(columns={"Text": "Sentence", "Ekman_Label": "Emotion"}, inplace=True)

# bỏ NA
for df in [dftrain, dfvalid, dftest]:
    df.dropna(subset=["Sentence", "Emotion"], inplace=True)

print(len(dftrain), len(dfvalid), len(dftest))
print(dftrain.columns)
print(dftrain.head(2))


11426 1583 3166
Index(['Sentence', 'Original_Sentiment', 'Emotion'], dtype='object')
                                         Sentence  Original_Sentiment  \
0                       slide giáo trình đầy đủ .                   2   
1  nhiệt tình giảng dạy , gần gũi với sinh viên .                   2   

     Emotion  
0  Happiness  
1  Happiness  


In [8]:
import numpy as np
import pandas as pd

def add_onto_stats(df, ontengine, text_col="Sentence", eps=1e-6):
    # trả về df mới có các cột thống kê ontology
    rows = []
    for text in df[text_col].astype(str).tolist():
        v = ontengine.getvector(text)
        av = np.abs(v)

        rows.append({
            "onto_any": float(av.sum() > eps),
            "onto_emotion": float(av[:7].sum() > eps),
            "onto_appraisal": float(av[7:16].sum() > eps),
            "onto_negcue": float(av[-2] > 0.5),          # vn[0]
            "onto_nnz": int((av > eps).sum()),
        })
    out = df.copy()
    out = pd.concat([out.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    return out

def coverage_by_label(df, label_col="Emotion"):
    g = df.groupby(label_col, dropna=False)
    rep = g.agg(
        N=(label_col, "size"),
        coverage_any_pct=("onto_any", "mean"),
        coverage_emotion_pct=("onto_emotion", "mean"),
        coverage_appraisal_pct=("onto_appraisal", "mean"),
        negcue_rate_pct=("onto_negcue", "mean"),
        avg_nnz_dims=("onto_nnz", "mean"),
    ).reset_index()

    # đổi mean -> %
    for c in ["coverage_any_pct","coverage_emotion_pct","coverage_appraisal_pct","negcue_rate_pct"]:
        rep[c] = (rep[c] * 100).round(3)

    rep["avg_nnz_dims"] = rep["avg_nnz_dims"].round(3)

    # sort: lớp nào ít mẫu lên trước hoặc theo negcue tùy bạn
    return rep.sort_values(["N"], ascending=True)

# --- chạy cho từng split ---
dftrain_s = add_onto_stats(dftrain, ontengine)
dfvalid_s = add_onto_stats(dfvalid, ontengine)  # nếu bạn có dfvalid
dftest_s  = add_onto_stats(dftest,  ontengine)

train_report = coverage_by_label(dftrain_s, label_col="Emotion")
val_report   = coverage_by_label(dfvalid_s, label_col="Emotion")
test_report  = coverage_by_label(dftest_s,  label_col="Emotion")

print("TRAIN\n", train_report.to_string(index=False))
print("\nVAL\n", val_report.to_string(index=False))
print("\nTEST\n", test_report.to_string(index=False))


TRAIN
   Emotion    N  coverage_any_pct  coverage_emotion_pct  coverage_appraisal_pct  negcue_rate_pct  avg_nnz_dims
     Fear   39            74.359                74.359                     0.0           15.385         2.590
 Surprise   42            92.857                83.333                     0.0           26.190         3.071
  Neutral  457            47.484                40.481                     0.0           15.755         1.024
  Disgust  620            85.000                71.774                     0.0           38.871         2.335
    Anger 2143            80.308                77.042                     0.0           16.146         2.144
  Sadness 2494            81.155                71.411                     0.0           28.027         2.176
Happiness 5631            80.465                79.950                     0.0            2.859         1.912

VAL
   Emotion   N  coverage_any_pct  coverage_emotion_pct  coverage_appraisal_pct  negcue_rate_pct  avg_nnz_dim

In [9]:
# =========================================================
# CELL: STRESS TEST LOGIC (V7.2)
# =========================================================
test_cases = [
    # 1. TEST PHỦ ĐỊNH + SCOPE (Liệu 'không' có xuyên qua 'hề' để bắt 'thích'?)
    ("tôi không hề thích nó", "Negation Scope Check"),
    
    # 2. TEST ĐẢO NGHĨA (Liệu 'Hạnh phúc' + 'Không' có thành 'Buồn'?)
    ("cảm thấy không vui chút nào", "Flip Logic Check"),
    
    # 3. TEST TỪ ĐA NGHĨA (Liệu 'chả' trong 'bún chả' có bị bắt nhầm là 'không'?)
    ("bún chả rất ngon", "Ambiguity Filter Check (Chả)"),
    
    # 4. TEST TỪ ĐA NGHĨA 2 (Liệu 'hay' có bị bắt nhầm là Vui?)
    ("làm hay nghỉ", "Ambiguity Filter Check (Hay)"),
    
    # 5. TEST CỤM TỪ CỐ ĐỊNH (MWE)
    ("thất vọng tràn trề", "MWE Recognition")
]

print(f"{'='*20} BÁO CÁO KẾT QUẢ TEST {'='*20}")

for text, desc in test_cases:
    print(f"\n🔹 CASE: '{text}' ({desc})")
    vec, tokens, trace = ontengine.getvector(text, debug=True)
    
    # In ra trace để xem Engine bắt từ nào
    print(f"   ► Tokens: {tokens}")
    if trace:
        print(f"   ► Matched: {[t['phrase'] for t in trace]}")
        # Check xem có Negation không
        is_neg = any(t['data']['isnegation'] for t in trace)
        print(f"   ► Detected Negation: {is_neg}")
    else:
        print("   ► Matched: [NONE]")
        
    # In ra các chiều vector có giá trị cao nhất
    labels = ontengine.ALLEMOTIONS
    emotions_vals = vec[:7] # 7 chiều đầu là Emotion
    max_idx = np.argmax(emotions_vals)
    
    if emotions_vals[max_idx] > 0:
        print(f"   ► DOMINANT EMOTION: {labels[max_idx]} (Score: {emotions_vals[max_idx]:.4f})")
    else:
        print(f"   ► DOMINANT EMOTION: [Neutral/None]")

print("\n" + "="*60)

==================== BÁO CÁO KẾT QUẢ TEST ====================

🔹 CASE: 'tôi không hề thích nó' (Negation Scope Check)
   ► Tokens: ['tôi', 'không', 'hề', 'thích', 'nó']
   ► Matched: ['không', 'thích']
   ► Detected Negation: True
   ► DOMINANT EMOTION: Happiness (Score: 0.4510)

🔹 CASE: 'cảm thấy không vui chút nào' (Flip Logic Check)
   ► Tokens: ['cảm thấy', 'không', 'vui', 'chút', 'nào']
   ► Matched: ['không', 'vui']
   ► Detected Negation: True
   ► DOMINANT EMOTION: Happiness (Score: 0.4941)

🔹 CASE: 'bún chả rất ngon' (Ambiguity Filter Check (Chả))
   ► Tokens: ['bún chả', 'rất', 'ngon']
   ► Matched: ['rất']
   ► Detected Negation: False
   ► DOMINANT EMOTION: Happiness (Score: 0.4189)

🔹 CASE: 'làm hay nghỉ' (Ambiguity Filter Check (Hay))
   ► Tokens: ['làm', 'hay', 'nghỉ']
   ► Matched: ['nghỉ']
   ► Detected Negation: False
   ► DOMINANT EMOTION: Sadness (Score: 0.3888)

🔹 CASE: 'thất vọng tràn trề' (MWE Recognition)
   ► Tokens: ['thất vọng', 'tràn trề']
   ► Matched: ['t

In [ ]:
# ---------------------------------------------------------
# CELL 6: MAIN EXPERIMENTS (5 CASES)
# ---------------------------------------------------------
NC = len(le.classes_)

# 1. Baseline
m1 = PhoBERT_Fusion_V2(NC, 'none')
m1 = train_experiment(m1, "Case 1: Baseline", epochs=15)



2026-01-08 04:21:55.369433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767846115.563752      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767846115.621281      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767846116.087596      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767846116.087660      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767846116.087663      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 Bắt đầu huấn luyện: Case 1: Baseline
   (Cấu hình: Max Epochs=15, Patience=3)


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Epoch 1/15 | Loss: 1.4835 | Val F1: 0.4450 ✅ New Best!
Epoch 2/15 | Loss: 1.1045 | Val F1: 0.5059 ✅ New Best!
Epoch 3/15 | Loss: 0.8524 | Val F1: 0.5711 ✅ New Best!
Epoch 4/15 | Loss: 0.6371 | Val F1: 0.4710 ⚠️ No improve (1/3)


In [ ]:
# 2. Raw + Concat (Best xAI)
m2 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=None)
m2 = train_experiment(m2, "Case 2: Raw (24d) + Concat", epochs=15)

# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Case 3: Raw (24d) + Gate", epochs=15)



In [ ]:
# 4. Dense (64d) + Concat (Trade-off xAI for Nonlinearity)
m4 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=64)
m4 = train_experiment(m4, "Case 4: Dense (64d) + Concat", epochs=15)



In [ ]:
# 5. Dense (64d) + Gate
m5 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=64)
m5 = train_experiment(m5, "Case 5: Dense (64d) + Gate", epochs=15)

In [ ]:
# ---------------------------------------------------------
# CELL 7: FINAL REPORT (5 MODELS)
# ---------------------------------------------------------
def gp(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            _, pr = torch.max(model(ids, m, o), 1)
            p.extend(pr.cpu().numpy()); t.extend(l.cpu().numpy())
    return t, p

y_true, p1 = gp(m1, test_loader)
_, p2 = gp(m2, test_loader)
_, p3 = gp(m3, test_loader)
_, p4 = gp(m4, test_loader)
_, p5 = gp(m5, test_loader)

res = {
    "Model": ["Baseline", "Raw+Concat (xAI)", "Raw+Gate", "Dense(64)+Concat", "Dense(64)+Gate"],
    "Ontology Dim": ["-", "24 (Raw)", "24 (Raw)", "64 (Dense)", "64 (Dense)"],
    "F1-Macro": [f1_score(y_true, p, average='macro') for p in [p1, p2, p3, p4, p5]],
    "Accuracy": [accuracy_score(y_true, p) for p in [p1, p2, p3, p4, p5]]
}

print("\n" + "="*50)
print("🏆 FINAL THESIS RESULTS")
print("="*50)
print(pd.DataFrame(res).round(4).to_markdown(index=False))

print("\n" + "="*50)
print("📑 DETAILED REPORT (CASE 5: DENSE + GATE)")
print("="*50)
print(classification_report(y_true, p5, target_names=le.classes_, digits=4))

In [ ]:
# ---------------------------------------------------------\
# CELL: THỰC NGHIỆM CHIỀU SÂU (DEEP ONTOLOGY PROJECTION)
# Architecture: Input(24) -> Dense(64) -> ReLU -> Dense(768) -> Fusion
# ---------------------------------------------------------\
import torch
import torch.nn as nn
from transformers import AutoModel

# --- [FIX LỖI] ĐỊNH NGHĨA NUM_CLASSES ---
try:
    # Cố gắng lấy từ LabelEncoder nếu đã chạy cell data
    NUM_CLASSES = len(le.classes_)
    print(f"✅ Đã tìm thấy {NUM_CLASSES} nhãn từ LabelEncoder.")
except NameError:
    # Fallback nếu chưa có biến le
    NUM_CLASSES = 7
    print("⚠️ Không tìm thấy biến 'le', sử dụng mặc định NUM_CLASSES = 7 (VSFC Standard).")

# 1. Định nghĩa lại Model Deep Ontology
class PhoBERT_DeepOntology(nn.Module):
    def __init__(self, n_classes, fusion_type='concat', ont_input_dim=24, hidden_dim=64):
        super(PhoBERT_DeepOntology, self).__init__()
        self.fusion_type = fusion_type
        
        # Backbone PhoBERT (768d)
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # --- KHỐI XỬ LÝ ONTOLOGY (24 -> 64 -> 768) ---
        self.ont_pipeline = nn.Sequential(
            # Bước 1: Nén/Biến đổi sơ cấp (24 -> 64)
            nn.Linear(ont_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            # Bước 2: Phóng chiếu sang không gian ngữ nghĩa (64 -> 768)
            nn.Linear(hidden_dim, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Cổng Fusion (Gate) - Nếu dùng
        if self.fusion_type == 'gate':
            self.gate_fc = nn.Linear(768 * 2, 768) 
        
        # Classifier Head
        # Nếu Concat: 768 (Text) + 768 (Ont) = 1536
        # Nếu Gate/Add: 768
        final_input_dim = 768 * 2 if self.fusion_type == 'concat' else 768
        self.fc = nn.Linear(final_input_dim, n_classes)

    def forward(self, input_ids, attention_mask, ontology_features):
        # 1. PhoBERT Forward
        _, text_emb = self.phobert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        text_emb = self.dropout(text_emb)
        
        # 2. Ontology Forward (24 -> 64 -> 768)
        ont_emb = self.ont_pipeline(ontology_features)
        
        # 3. Fusion Strategy
        if self.fusion_type == 'concat':
            # Nối đuôi: [Batch, 1536]
            combined = torch.cat([text_emb, ont_emb], dim=1)
            logits = self.fc(combined)
            
        elif self.fusion_type == 'gate':
            # Cổng trọng số động
            concat_feat = torch.cat([text_emb, ont_emb], dim=1)
            z = torch.sigmoid(self.gate_fc(concat_feat))
            fused_emb = (z * text_emb) + ((1 - z) * ont_emb)
            logits = self.fc(fused_emb)
            
        else: # 'none' hoặc 'add'
            logits = self.fc(text_emb + ont_emb)
            
        return logits

# ---------------------------------------------------------
# CHẠY THỰC NGHIỆM
# ---------------------------------------------------------

# Lấy kích thước vector thực tế từ engine (thường là 24)
ONT_DIM = 24 

# === CASE A: DEEP PROJECTION + CONCAT ===
print("\n" + "="*40)
print("🧪 CASE A: Deep Ontology (24->64->768) + CONCAT")
print("="*40)
model_deep_concat = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='concat', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=64
)
# In cấu trúc để kiểm tra
print(model_deep_concat.ont_pipeline) 

model_deep_concat = train_experiment(
    model_deep_concat, 
    "Deep Ont (64d) + Concat", 
    epochs=15, 
    patience=4
)
acc_a, f1_a = evaluate(model_deep_concat, test_loader)


# === CASE B: DEEP PROJECTION + GATE (PROPOSED) ===
print("\n" + "="*40)
print("🧪 CASE B: Deep Ontology (24->64->768) + GATE")
print("="*40)
model_deep_gate = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='gate', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=64
)

model_deep_gate = train_experiment(
    model_deep_gate, 
    "Deep Ont (64d) + Gate", 
    epochs=15, 
    patience=4
)
acc_b, f1_b = evaluate(model_deep_gate, test_loader)

# === TỔNG HỢP KẾT QUẢ MỚI ===
print(f"\n📊 KẾT QUẢ KIẾN TRÚC 24->64->768:")
print(f"1. Concat: F1-Macro = {f1_a:.4f} | Acc = {acc_a:.4f}")
print(f"2. Gate  : F1-Macro = {f1_b:.4f} | Acc = {acc_b:.4f}")

In [ ]:
class PhoBERT_WideProjection(nn.Module):
    def __init__(self, n_classes, fusion_type='gate', ont_input_dim=24):
        super(PhoBERT_WideProjection, self).__init__()
        self.fusion_type = fusion_type
        
        # Backbone PhoBERT
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        self.dropout = nn.Dropout(p=0.3)
        
        # --- KHỐI WIDE & DEEP PROJECTION (24 -> 768 -> 768) ---
        self.ont_pipeline = nn.Sequential(
            # Phóng chiếu từ vector thô lên không gian của BERT
            nn.Linear(ont_input_dim, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            
            # Lớp xử lý chiều sâu tăng cường phi tuyến tính
            nn.Linear(768, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Fusion Head
        if self.fusion_type == 'gate':
            self.gate_fc = nn.Linear(768 * 2, 768)
            self.fc = nn.Linear(768, n_classes)
        elif self.fusion_type == 'concat':
            self.fc = nn.Linear(768 * 2, n_classes)
        else: # none
            self.fc = nn.Linear(768, n_classes)

    def forward(self, input_ids, attention_mask, ontology_features):
        # 1. PhoBERT features (768d)
        _, text_emb = self.phobert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        text_emb = self.dropout(text_emb)
        
        # 2. Ontology Wide Projection (24 -> 768 -> 768)
        ont_emb = self.ont_pipeline(ontology_features)
        
        # 3. Fusion
        if self.fusion_type == 'concat':
            combined = torch.cat([text_emb, ont_emb], dim=1)
            return self.fc(combined)
            
        elif self.fusion_type == 'gate':
            # Dynamic gating between BERT and Projected Ontology
            gate_input = torch.cat([text_emb, ont_emb], dim=1)
            z = torch.sigmoid(self.gate_fc(gate_input))
            fused = (z * text_emb) + ((1 - z) * ont_emb)
            return self.fc(fused)
            
        return self.fc(text_emb)

In [ ]:
# === CASE C: WIDE PROJECTION (768d) + CONCAT ===
print("\n" + "="*40)
print("🧪 CASE C: Wide Projection (24->768->768) + CONCAT")
print("="*40)
model_wide_concat = PhoBERT_WideProjection(n_classes=NUM_CLASSES, fusion_type='concat')

model_wide_concat = train_experiment(
    model_wide_concat, 
    "Wide Ont (768d) + Concat", 
    epochs=15, 
    patience=4
)
acc_c, f1_c = evaluate(model_wide_concat, test_loader)

# === CASE D: WIDE PROJECTION (768d) + GATE ===
print("\n" + "="*40)
print("🧪 CASE D: Wide Projection (24->768->768) + GATE")
print("="*40)
model_wide_gate = PhoBERT_WideProjection(n_classes=NUM_CLASSES, fusion_type='gate')

model_wide_gate = train_experiment(
    model_wide_gate, 
    "Wide Ont (768d) + Gate", 
    epochs=15, 
    patience=4
)
acc_d, f1_d = evaluate(model_wide_gate, test_loader)

# --- TỔNG KẾT ---
print(f"\n📊 KẾT QUẢ KIẾN TRÚC WIDE PROJECTION (24->768->768):")
print(f"1. Wide + Concat: F1-Macro = {f1_c:.4f} | Acc = {acc_c:.4f}")
print(f"2. Wide + Gate  : F1-Macro = {f1_d:.4f} | Acc = {acc_d:.4f}")